# 🌡 TP 02 — Régulation de la Température par Logique Floue
**INTeK — Institut National des Technologies et des Sciences du Kef**  
Réalisé par : HAMMEDI NOURHEN & BOUALI NADA  
Filière : Génie Biomédical

---
Ce notebook implémente un système de régulation floue pour un **incubateur médical**.
Les entrées sont :
- **Température externe** (°C) — ambiance autour de l'incubateur
- **Température cutanée** (°C) — température de la peau du nourrisson

La sortie est :
- **Puissance du chauffage** (%) — entre 0 et 100 %

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1 & 2 — Définition des univers et fonctions d'appartenance

In [ ]:
# ── Univers de discours ───────────────────────────────────────────────────────
externe = np.arange(0, 51, 0.5)    # Température externe : 0 → 50 °C
interne = np.arange(34, 41.1, 0.1) # Température cutanée : 34 → 41 °C
puiss   = np.arange(0, 101, 1)     # Puissance : 0 → 100 %

# ── Variables linguistiques ───────────────────────────────────────────────────
T_externe = ctrl.Antecedent(externe, 'temperature_externe')
T_interne = ctrl.Antecedent(interne,  'temperature_cutanee')
puissance = ctrl.Consequent(puiss,    'puissance')

# ── Fonctions d'appartenance (TRAPÉZOIDALES — section 1-2 du TP) ─────────────
# Température externe
T_externe['froid']  = fuzz.trapmf(externe, [0,  3, 12, 15])
T_externe['chaud']  = fuzz.trapmf(externe, [20, 25, 45, 50])

# Température cutanée
T_interne['hypothermie'] = fuzz.trapmf(interne, [34,   34.5, 36,   36.5])
T_interne['normal']      = fuzz.trapmf(interne, [36,   36.5, 37.5, 38  ])
T_interne['fievre']      = fuzz.trapmf(interne, [37.5, 38,   40.5, 41  ])

# Puissance (fonctions triangulaires)
puissance['Faible']   = fuzz.trimf(puiss, [0,  20, 40 ])
puissance['Moyenne']  = fuzz.trimf(puiss, [35, 55, 75 ])
puissance['Maximale'] = fuzz.trimf(puiss, [70, 85, 100])

print('✅ Univers et fonctions d\'appartenance définis.')

In [ ]:
# ── Tracé des fonctions d'appartenance ───────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
fig.suptitle('Fonctions d\'appartenance — Système Flou Incubateur', fontsize=14, fontweight='bold')

# Température externe
axes[0].plot(externe, T_externe['froid'].mf,  label='froid',  color='#3A86FF', lw=2)
axes[0].plot(externe, T_externe['chaud'].mf,  label='chaud',  color='#FF6B6B', lw=2)
axes[0].set_title('Température externe (°C)'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Température cutanée
axes[1].plot(interne, T_interne['hypothermie'].mf, label='hypothermie', color='#06D6A0', lw=2)
axes[1].plot(interne, T_interne['normal'].mf,      label='normal',      color='#FFD166', lw=2)
axes[1].plot(interne, T_interne['fievre'].mf,      label='fièvre',      color='#EF476F', lw=2)
axes[1].set_title('Température cutanée (°C)'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Puissance
axes[2].plot(puiss, puissance['Faible'].mf,   label='Faible',   color='#118AB2', lw=2)
axes[2].plot(puiss, puissance['Moyenne'].mf,  label='Moyenne',  color='#FF9F1C', lw=2)
axes[2].plot(puiss, puissance['Maximale'].mf, label='Maximale', color='#2EC4B6', lw=2)
axes[2].set_title('Puissance du chauffage (%)'); axes[2].legend(); axes[2].grid(alpha=0.3)

for ax in axes:
    ax.set_ylabel("Degré d'appartenance"); ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.savefig('figures/membership_trapezoidal.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 — Règles d'inférence

In [ ]:
# ── 6 règles d'inférence ──────────────────────────────────────────────────────
rule1 = ctrl.Rule(T_externe['froid'] & T_interne['hypothermie'], puissance['Maximale'])
rule2 = ctrl.Rule(T_externe['froid'] & T_interne['normal'],      puissance['Moyenne'])
rule3 = ctrl.Rule(T_externe['froid'] & T_interne['fievre'],      puissance['Faible'])
rule4 = ctrl.Rule(T_externe['chaud'] & T_interne['hypothermie'], puissance['Moyenne'])
rule5 = ctrl.Rule(T_externe['chaud'] & T_interne['normal'],      puissance['Faible'])
rule6 = ctrl.Rule(T_externe['chaud'] & T_interne['fievre'],      puissance['Faible'])

print('Règles définies :')
print('  R1: froid ∧ hypothermie → Maximale')
print('  R2: froid ∧ normal      → Moyenne')
print('  R3: froid ∧ fièvre      → Faible')
print('  R4: chaud ∧ hypothermie → Moyenne')
print('  R5: chaud ∧ normal      → Faible')
print('  R6: chaud ∧ fièvre      → Faible')

## 4 & 5 — Implémentation et Tests

In [ ]:
# ── Construction du système ───────────────────────────────────────────────────
systeme = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6])
sim = ctrl.ControlSystemSimulation(systeme)

# ── Tests (section 5) ─────────────────────────────────────────────────────────
Tests = [
    (10, 35, 'Froid + Hypothermie'),
    (15, 37, 'Froid + Normal'),
    (25, 39, 'Chaud + Fièvre'),
]

print('='*55)
print('  Tests — Système complet (Centroid)')
print('='*55)
for Te, Ti, desc in Tests:
    sim.input['temperature_externe']  = Te
    sim.input['temperature_cutanee'] = Ti
    sim.compute()
    p = sim.output['puissance']
    print(f'  T_ext={Te:>3}°C  T_int={Ti:.1f}°C  →  Puissance = {p:.2f}%   [{desc}]')

In [ ]:
# ── Visualisation de la sortie (view) ────────────────────────────────────────
sim.input['temperature_externe']  = 10
sim.input['temperature_cutanee'] = 35
sim.compute()
puissance.view(sim=sim)
plt.suptitle('Visualisation défuzzification — T_ext=10°C, T_int=35°C')
plt.show()

## 6 — Comparaison des méthodes de défuzzification : MOM vs Centroid

In [ ]:
import skfuzzy as fuzz

def make_sim(method):
    """Reconstruit le système avec la méthode de défuzzification choisie."""
    T_ext2 = ctrl.Antecedent(externe, 'temperature_externe')
    T_int2 = ctrl.Antecedent(interne, 'temperature_cutanee')
    puis2  = ctrl.Consequent(puiss, 'puissance', defuzzify_method=method)
    T_ext2['froid']  = fuzz.trapmf(externe, [0,  3, 12, 15])
    T_ext2['chaud']  = fuzz.trapmf(externe, [20, 25, 45, 50])
    T_int2['hypothermie'] = fuzz.trapmf(interne, [34, 34.5, 36, 36.5])
    T_int2['normal']      = fuzz.trapmf(interne, [36, 36.5, 37.5, 38])
    T_int2['fievre']      = fuzz.trapmf(interne, [37.5, 38, 40.5, 41])
    puis2['Faible']   = fuzz.trimf(puiss, [0,  20, 40])
    puis2['Moyenne']  = fuzz.trimf(puiss, [35, 55, 75])
    puis2['Maximale'] = fuzz.trimf(puiss, [70, 85, 100])
    rules = [
        ctrl.Rule(T_ext2['froid'] & T_int2['hypothermie'], puis2['Maximale']),
        ctrl.Rule(T_ext2['froid'] & T_int2['normal'],      puis2['Moyenne']),
        ctrl.Rule(T_ext2['froid'] & T_int2['fievre'],      puis2['Faible']),
        ctrl.Rule(T_ext2['chaud'] & T_int2['hypothermie'], puis2['Moyenne']),
        ctrl.Rule(T_ext2['chaud'] & T_int2['normal'],      puis2['Faible']),
        ctrl.Rule(T_ext2['chaud'] & T_int2['fievre'],      puis2['Faible']),
    ]
    return ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

test_pairs = [(10, 35), (20, 37), (35, 39)]
methods = [('centroid', 'Centre de Gravité'), ('mom', 'MOM')]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Comparaison Centroid vs MOM', fontsize=13, fontweight='bold')

for ax, (method, name) in zip(axes, methods):
    s = make_sim(method)
    results = []
    print(f'\n── {name} ──')
    for Te, Ti in test_pairs:
        s.input['temperature_externe']  = Te
        s.input['temperature_cutanee'] = Ti
        s.compute()
        p = s.output['puissance']
        print(f'  T_ext={Te}°C  T_int={Ti}°C  →  {p:.2f}%')
        results.append(p)
    labels = [f'({te}°,{ti}°)' for te, ti in test_pairs]
    bars = ax.bar(labels, results, color=['#3A86FF','#FF6B6B','#06D6A0'], edgecolor='white')
    ax.set_title(name); ax.set_ylabel('Puissance (%)'); ax.set_ylim(0,110); ax.grid(axis='y',alpha=0.3)
    for b, v in zip(bars, results):
        ax.text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/comparison_defuzz.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Modification des fonctions d'appartenance (Gaussiennes)

In [ ]:
# ── Reconstruction avec fonctions gaussiennes ────────────────────────────────
T_ext_g = ctrl.Antecedent(externe, 'temperature_externe')
T_int_g = ctrl.Antecedent(interne, 'temperature_cutanee')
puis_g  = ctrl.Consequent(puiss,   'puissance')

T_ext_g['froid']  = fuzz.gaussmf(externe, mean=5,  sigma=10)
T_ext_g['chaud']  = fuzz.gaussmf(externe, mean=35, sigma=10)
T_int_g['hypothermie'] = fuzz.gaussmf(interne, mean=34.5, sigma=0.6)
T_int_g['normal']      = fuzz.gaussmf(interne, mean=37.0, sigma=0.5)
T_int_g['fievre']      = fuzz.gaussmf(interne, mean=39.5, sigma=0.7)
puis_g['Faible']   = fuzz.gaussmf(puiss, mean=20, sigma=10)
puis_g['Moyenne']  = fuzz.gaussmf(puiss, mean=55, sigma=12)
puis_g['Maximale'] = fuzz.gaussmf(puiss, mean=85, sigma=10)

rules_g = [
    ctrl.Rule(T_ext_g['froid'] & T_int_g['hypothermie'], puis_g['Maximale']),
    ctrl.Rule(T_ext_g['froid'] & T_int_g['normal'],      puis_g['Moyenne']),
    ctrl.Rule(T_ext_g['froid'] & T_int_g['fievre'],      puis_g['Faible']),
    ctrl.Rule(T_ext_g['chaud'] & T_int_g['hypothermie'], puis_g['Moyenne']),
    ctrl.Rule(T_ext_g['chaud'] & T_int_g['normal'],      puis_g['Faible']),
    ctrl.Rule(T_ext_g['chaud'] & T_int_g['fievre'],      puis_g['Faible']),
]
sim_g = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules_g))

# Tracé
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
fig.suptitle('Fonctions d\'appartenance Gaussiennes', fontsize=14, fontweight='bold')
axes[0].plot(externe, T_ext_g['froid'].mf,  label='froid',  color='#3A86FF', lw=2)
axes[0].plot(externe, T_ext_g['chaud'].mf,  label='chaud',  color='#FF6B6B', lw=2)
axes[0].set_title('Température externe'); axes[0].legend(); axes[0].grid(alpha=0.3)
for label, color in [('hypothermie','#06D6A0'),('normal','#FFD166'),('fievre','#EF476F')]:
    axes[1].plot(interne, T_int_g[label].mf, label=label, color=color, lw=2)
axes[1].set_title('Température cutanée'); axes[1].legend(); axes[1].grid(alpha=0.3)
for label, color in [('Faible','#118AB2'),('Moyenne','#FF9F1C'),('Maximale','#2EC4B6')]:
    axes[2].plot(puiss, puis_g[label].mf, label=label, color=color, lw=2)
axes[2].set_title('Puissance'); axes[2].legend(); axes[2].grid(alpha=0.3)
for ax in axes:
    ax.set_ylabel("Degré d'appartenance"); ax.set_ylim(-0.05,1.1)
plt.tight_layout()
plt.savefig('figures/membership_gaussian.png', dpi=150, bbox_inches='tight')
plt.show()

# Tests gaussiens
print('\n── Tests Gaussiens ──')
for Te, Ti, desc in [(10,35,'Froid+Hypo'),(15,37,'Froid+Normal'),(25,39,'Chaud+Fièvre')]:
    sim_g.input['temperature_externe']  = Te
    sim_g.input['temperature_cutanee'] = Ti
    sim_g.compute()
    print(f'  T_ext={Te}°C T_int={Ti}°C → {sim_g.output["puissance"]:.2f}%  [{desc}]')

## 8 — Suppression de règles d'inférence

In [ ]:
from fuzzy_system import build_partial_system, run_tests

print('Cas 1 — Suppression règles 2 & 4 :')
sim_p1, *_ = build_partial_system(rules_to_remove=[2, 4])
run_tests(sim_p1, label='Règles 2 & 4 supprimées')

print('\nCas 2 — Suppression règles 3 & 5 :')
sim_p2, *_ = build_partial_system(rules_to_remove=[3, 5])
run_tests(sim_p2, label='Règles 3 & 5 supprimées')

## 9 — Interface graphique Tkinter
```bash
python interface.py
```